# Sigiriya Hybrid GraphRAG — Optimized Local Retrieval Evaluation (32 Sinhala Questions)

**Before SinLlama; VS Code + Neo4j Desktop only.** Uses the group's original functions and the existing Sigiriya graph. No rebuild, no graph writes, no model fine-tuning.

**Faster, easier-to-debug changes:**
- Neo4j connectivity is checked *before* loading E5; visible progress messages.
- E5 is initialized once per kernel, and repeated identical query embeddings are cached in RAM.
- A 5-question trial is the default; increase `RUN_COUNT` to 20 or 32 after checking results.
- Results are saved after **each question**, so interrupted runs can resume.
- Each operation retains the group's original hybrid retrieval and answer-selection pipeline. Speedup is not guaranteed where original algorithms use many distinct queries.

Use a `.venv` notebook kernel with `neo4j`, `sentence-transformers`, `pandas`, `ipykernel`, and `torch` installed. Start the **Sigiriya** Neo4j instance before running. The notebook reads only and does not use SinLlama.

## 1 — Requirements

Start the **Sigiriya Neo4j Desktop instance** and confirm it is the one listening on `bolt://127.0.0.1:7687`. Python environment dependencies: `neo4j`, `sentence-transformers`, `torch`, `pandas`, `ipykernel` (install via your root `requirements.txt`). Run in VS Code; do not paste your password into code or source control.

In [ ]:
from pathlib import Path
import os, sys, json, csv, time, getpass, importlib
import pandas as pd
from IPython.display import display
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer

# Locate group's original Python files relative to VS Code's working directory.
starts = [Path.cwd(), *list(Path.cwd().parents)[:4]]
locations = []
for root in starts:
    locations.extend([
        root, root / "original_group_work",
        root / "original_group_work" / "Sigiriya_Hybrid_GraphRAG_Stage2",
        root / "Sigiriya_Hybrid_GraphRAG_Stage2"
    ])
GROUP_DIR = next((p.resolve() for p in locations if (p / "Sigiriya_GraphRAG_Stage2.py").is_file()), None)
if GROUP_DIR is None:
    raise FileNotFoundError("Cannot find Sigiriya_GraphRAG_Stage2.py. Check original_group_work folder and notebook working directory.")
sys.path.insert(0, str(GROUP_DIR))
PROJECT_ROOT = next((p for p in [GROUP_DIR.parent, GROUP_DIR.parent.parent] if (p / 'results').is_dir()), GROUP_DIR.parent)
OUTPUT_DIR = PROJECT_ROOT / "results" / "retrieval"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Python:', sys.executable)
print('Group scripts:', GROUP_DIR)
print('Results folder:', OUTPUT_DIR)

## 2 — Connect to local Neo4j and load E5 once

Run once per kernel. Prints checkpoints and times. If the E5 files are cached, you can set `LOCAL_FILES_ONLY=True` to prohibit downloads. Leave it `False` if the model has not been completely cached. Do not run this cell again during the 32-question batch.

In [ ]:
import torch
from functools import lru_cache
import numpy as np
from getpass import getpass

print('Importing group retrieval modules...', flush=True)
import Sigiriya_Multi_Query_Retrieval as multi
import Sigiriya_GraphRAG_Stage2 as stage2
import Sigiriya_GraphRAG_Stage2_Answer_Refinement as refinement
from Sigiriya_Retrieval_Improvement_Batch_Reviewed import load_sources

URI = os.getenv('NEO4J_URI', 'bolt://127.0.0.1:7687')
USERNAME = os.getenv('NEO4J_USERNAME', 'neo4j')
DATABASE = os.getenv('NEO4J_DATABASE', 'neo4j')
if 'driver' not in globals():
    PASSWORD = os.getenv('NEO4J_PASSWORD') or getpass('Sigiriya Neo4j password: ')
    driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD), connection_timeout=10)

print('Checking Neo4j connectivity...', flush=True)
driver.verify_connectivity()
with driver.session(database=DATABASE) as session:
    rows = session.run("MATCH (c:Chunk {destination:'Sigiriya'}) RETURN count(c) AS n, count(c.embedding) AS vectors").single()
    entities = session.run("MATCH (e:TourismEntity {scope:'Sigiriya'}) RETURN count(e) AS n").single()['n']
    indexes = {r['name']: r['state'] for r in session.run('SHOW VECTOR INDEXES YIELD name,state RETURN name,state')}
assert rows['n'] == 22 and rows['vectors'] == 22 and entities == 49, (rows['n'], rows['vectors'], entities)
for name in (multi.TEXT_INDEX, multi.HEADING_INDEX):
    assert indexes.get(name) == 'ONLINE', f'Index {name} not ONLINE: {indexes.get(name)}'
print(f'Neo4j OK — {rows["n"]} chunks, {entities} entities, indexes ONLINE.', flush=True)

# Uses the original two-index search, but with the legacy Neo4j vector procedure.
# Do not modify the original group source files.
def local_vector_search(session, index_name, vector):
    if index_name not in (multi.TEXT_INDEX, multi.HEADING_INDEX):
        raise ValueError(f'Unexpected vector index: {index_name}')
    q = ("CALL db.index.vector.queryNodes($index, $k, $vector) "
         "YIELD node, score WHERE node:Chunk AND node.destination='Sigiriya' "
         "RETURN node.id AS chunk_id, score ORDER BY score DESC")
    return [r['chunk_id'] for r in session.run(q, index=index_name, k=22, vector=vector)]
multi.search_index = local_vector_search

LOCAL_FILES_ONLY = False  # Set True only if the E5 model is fully downloaded.
if 'model' not in globals():
    print('Loading E5 on CPU (first load can be slow)...', flush=True)
    t0 = time.perf_counter()
    model = SentenceTransformer(multi.MODEL_NAME, device='cpu', local_files_only=LOCAL_FILES_ONLY)
    print(f'E5 loaded in {time.perf_counter()-t0:.1f}s', flush=True)
else:
    print('Reusing E5 already in memory.', flush=True)

# Repeated calls to fused_search/quality_answer can encode identical questions.
# Cache ONLY normal single-string inputs and return the same NumPy shape expected by .tolist().
if not hasattr(model, '_sigiriya_uncached_encode'):
    model._sigiriya_uncached_encode = model.encode
    @lru_cache(maxsize=512)
    def _cached_encode(sentence, normalized):
        return model._sigiriya_uncached_encode(sentence, normalize_embeddings=normalized)
    def _encode_with_cache(sentences, *args, **kwargs):
        if isinstance(sentences, str) and not args and set(kwargs).issubset({'normalize_embeddings'}):
            return _cached_encode(sentences, kwargs.get('normalize_embeddings', False)).copy()
        return model._sigiriya_uncached_encode(sentences, *args, **kwargs)
    model.encode = _encode_with_cache
print('Query embedding cache: enabled (identical inputs only).', flush=True)

manifest = stage2.load_manifest(GROUP_DIR / 'Sigiriya_KG_Expansion_Manifest.json')
with driver.session(database=DATABASE) as session:
    headings = {r['chunk_id']:r['heading'] for r in session.run(multi.HEADINGS_QUERY)}
    sources = load_sources(session)
print('Ready to test! Source chunks:', len(sources), flush=True)

## 3 — Question set (32 tests)

The first **22** questions cover all sections of your uploaded Sinhala knowledge base. The next **7** require multiple sections. The final **3** are not answerable with reliable static document evidence. For section-based questions, `expected_ids` identifies relevant source chunks (e.g. `sigiriya_006`), **not a predefined model answer**.

**Important:** Current prices and opening times in the document are historical/source-supplied information; the system must **not** present them as today's verified values. Labels are based on the provided document, not independently verified facts about Sigiriya.

In [ ]:
TESTS = [{'id': 'Q01', 'type': 'single', 'question': 'සීගිරිය පිහිටා ඇත්තේ කුමන පළාතේ සහ දිස්ත්\u200dරික්කයේද?', 'expected_ids': ['sigiriya_001'], 'answerability': 'supported'}, {'id': 'Q02', 'type': 'single', 'question': 'සීගිරිය රාජකීය අගනුවරක් ලෙස සංවර්ධනය කළේ කවර රජුද?', 'expected_ids': ['sigiriya_002'], 'answerability': 'supported'}, {'id': 'Q03', 'type': 'single', 'question': 'සීගිරි පර්වතයේ මුදුනේ සිට දැකගත හැක්කේ මොනවාද?', 'expected_ids': ['sigiriya_003'], 'answerability': 'supported'}, {'id': 'Q04', 'type': 'single', 'question': 'සීගිරි බිතුසිතුවම්වල දැක්වෙන්නේ කවර රූපද?', 'expected_ids': ['sigiriya_004'], 'answerability': 'supported'}, {'id': 'Q05', 'type': 'single', 'question': 'කැටපත් පවුර මත පැරණි සංචාරකයන් ලියා තිබුණේ මොනවාද?', 'expected_ids': ['sigiriya_005'], 'answerability': 'supported'}, {'id': 'Q06', 'type': 'single', 'question': 'සීගිරියේ ජල උද්\u200dයානවල ජලය පාලනයට භාවිත කළේ මොනවාද?', 'expected_ids': ['sigiriya_006'], 'answerability': 'supported'}, {'id': 'Q07', 'type': 'single', 'question': 'සීගිරියේ දිය උල්පත් ක්\u200dරියාත්මක වීමට භාවිත වූ බලවේග මොනවාද?', 'expected_ids': ['sigiriya_007'], 'answerability': 'supported'}, {'id': 'Q08', 'type': 'single', 'question': 'සීගිරියේ ගල් උද්\u200dයානයේ දැකගත හැකි අංග මොනවාද?', 'expected_ids': ['sigiriya_008'], 'answerability': 'supported'}, {'id': 'Q09', 'type': 'single', 'question': 'නාග පෙන ගුහාවට එම නම ලැබුණේ ඇයි?', 'expected_ids': ['sigiriya_009'], 'answerability': 'supported'}, {'id': 'Q10', 'type': 'single', 'question': 'සීගිරි කෞතුකාගාරයෙන් දැනගත හැකි කරුණු මොනවාද?', 'expected_ids': ['sigiriya_010'], 'answerability': 'supported'}, {'id': 'Q11', 'type': 'single', 'question': 'ලේඛනයේ සඳහන් දකුණු ආසියානු සංචාරක ප්\u200dරවේශ ගාස්තුව කීයද?', 'expected_ids': ['sigiriya_011'], 'answerability': 'supported'}, {'id': 'Q12', 'type': 'single', 'question': 'සීගිරියේදී සංචාරකයන්ට කළ හැකි ක්\u200dරියාකාරකම් තුනක් මොනවාද?', 'expected_ids': ['sigiriya_012'], 'answerability': 'supported'}, {'id': 'Q13', 'type': 'single', 'question': 'පිදුරංගල නැගීමේදී අවසාන කොටසේ අවධානය යොමු කළ යුත්තේ මොනවාටද?', 'expected_ids': ['sigiriya_013'], 'answerability': 'supported'}, {'id': 'Q14', 'type': 'single', 'question': 'Mapagala පිහිටා ඇත්තේ සීගිරියට කුමන දිශාවෙන්ද?', 'expected_ids': ['sigiriya_014'], 'answerability': 'supported'}, {'id': 'Q15', 'type': 'single', 'question': 'සීගිරිය ආසන්නයේ වනජීවී සෆාරි සඳහා සඳහන් ජාතික වනෝද්\u200dයාන මොනවාද?', 'expected_ids': ['sigiriya_015'], 'answerability': 'supported'}, {'id': 'Q16', 'type': 'single', 'question': 'මහනුවර සිට සීගිරියට යාමට ලේඛනයේ සඳහන් මාර්ගය කුමක්ද?', 'expected_ids': ['sigiriya_016'], 'answerability': 'supported'}, {'id': 'Q17', 'type': 'single', 'question': 'සීගිරිය පර්වතය තරණය කිරීමට පෙර රැගෙන යා යුතු වැදගත් දෙය කුමක්ද?', 'expected_ids': ['sigiriya_017'], 'answerability': 'supported'}, {'id': 'Q18', 'type': 'single', 'question': 'සීගිරිය හා හබරණ අවට සඳහන් නවාතැන් වර්ග මොනවාද?', 'expected_ids': ['sigiriya_018'], 'answerability': 'supported'}, {'id': 'Q19', 'type': 'single', 'question': 'සීගිරිය නැරඹීමට උදෑසන පැමිණීම සුදුසු වන්නේ ඇයි?', 'expected_ids': ['sigiriya_019'], 'answerability': 'supported'}, {'id': 'Q20', 'type': 'single', 'question': 'වැසි සහිත දිනක සීගිරිය තරණය කිරීමේදී ඇති අවදානම් මොනවාද?', 'expected_ids': ['sigiriya_020'], 'answerability': 'supported'}, {'id': 'Q21', 'type': 'single', 'question': 'සීගිරි පර්වතයේ බඹර කූඩු අසල හැසිරිය යුත්තේ කෙසේද?', 'expected_ids': ['sigiriya_021'], 'answerability': 'supported'}, {'id': 'Q22', 'type': 'single', 'question': 'සීගිරියේ පුරාවිද්\u200dයාත්මක උරුමය රැක ගැනීමට සංචාරකයන් කළ යුතු දේ මොනවාද?', 'expected_ids': ['sigiriya_022'], 'answerability': 'supported'}, {'id': 'Q23', 'type': 'multi', 'question': 'සීගිරියේ ජල උද්\u200dයාන සහ දිය උල්පත් උද්\u200dයානයේ ජල පද්ධති විස්තර කරන්න.', 'expected_ids': ['sigiriya_006', 'sigiriya_007'], 'answerability': 'supported'}, {'id': 'Q24', 'type': 'multi', 'question': 'සීගිරි බිතුසිතුවම් සහ කැටපත් පවුරේ ඓතිහාසික වැදගත්කම පැහැදිලි කරන්න.', 'expected_ids': ['sigiriya_004', 'sigiriya_005'], 'answerability': 'supported'}, {'id': 'Q25', 'type': 'multi', 'question': 'සීගිරියේ ජල උද්\u200dයානය සහ ගල් උද්\u200dයානය අතර වෙනස්කම් මොනවාද?', 'expected_ids': ['sigiriya_006', 'sigiriya_008'], 'answerability': 'supported'}, {'id': 'Q26', 'type': 'multi', 'question': 'සීගිරිය සහ පිදුරංගල තරණය කිරීමේදී අවශ්\u200dය සූදානම ගැන කියන්න.', 'expected_ids': ['sigiriya_003', 'sigiriya_013', 'sigiriya_020'], 'answerability': 'supported'}, {'id': 'Q27', 'type': 'multi', 'question': 'සීගිරියට යන මාර්ග සහ නැරඹීමට සුදුසු වේලාව පැහැදිලි කරන්න.', 'expected_ids': ['sigiriya_016', 'sigiriya_019'], 'answerability': 'supported'}, {'id': 'Q28', 'type': 'multi', 'question': 'සීගිරියේ බඹර අනතුරු සහ ඓතිහාසික ස්ථාන රැකගැනීම ගැන උපදෙස් දෙන්න.', 'expected_ids': ['sigiriya_021', 'sigiriya_022'], 'answerability': 'supported'}, {'id': 'Q29', 'type': 'multi', 'question': 'සීගිරිය අවට නවාතැන් සහ වනජීවී සෆාරි අවස්ථා ගැන විස්තර කරන්න.', 'expected_ids': ['sigiriya_018', 'sigiriya_015'], 'answerability': 'supported'}, {'id': 'Q30', 'type': 'unsupported', 'question': 'සීගිරියේ අද ප්\u200dරවේශ පත්\u200dර මිල කීයද?', 'expected_ids': [], 'answerability': 'live_unsupported'}, {'id': 'Q31', 'type': 'unsupported', 'question': 'සීගිරියේ මේ මොහොතේ කාලගුණය කෙසේද?', 'expected_ids': [], 'answerability': 'live_unsupported'}, {'id': 'Q32', 'type': 'unsupported', 'question': 'සීගිරියේ 2035 වසරේ ප්\u200dරවේශ ගාස්තුව කීයද?', 'expected_ids': [], 'answerability': 'unsupported'}]

question_df = pd.DataFrame(TESTS)
display(question_df[['id','type','question','expected_ids','answerability']])
print('Questions:',len(TESTS), '| single:',sum(t['type']=='single' for t in TESTS),'| multi:',sum(t['type']=='multi' for t in TESTS),'| unsupported:',sum(t['type']=='unsupported' for t in TESTS))

## 4 — Define retrieval function (no automatic test)

Uses the **same complete hybrid retrieval and extractive-answer pipeline** as the original notebook. The function is defined here; nothing is retrieved until you run the batch in Step 5.

In [ ]:
def retrieve_one(test):
    q = test['question']
    t0 = time.perf_counter()
    with driver.session(database=DATABASE) as session:
        # Grounded extractive answer (same as your original export script)
        extracted = refinement.quality_answer(q, session, model, sources, headings)
        status = extracted.get('status', 'UNKNOWN')
        if status == 'LIVE_DATA_UNSUPPORTED':
            original, multi_rank, hybrid, paths, anchors = [], [], [], [], []
        else:
            original = multi.fused_search(session, model, q)
            _, multi_rank = multi.multi_query_search(session, model, q, headings)
            anchors = refinement.resolve_anchors(session, q, manifest['entities'], original)
            paths = refinement.graph_paths(session, q, manifest['entities'], anchors, limit=12)
            hybrid, _ = refinement.rank_chunks(original, multi_rank, paths)
        expected = set(test['expected_ids'])
        top3 = hybrid[:3]
        hits = expected.intersection(top3)
        return {
            'id': test['id'], 'type': test['type'], 'question': q,
            'answerability': test['answerability'], 'expected_ids': test['expected_ids'],
            'status': status, 'extractive_answer': extracted.get('answer', ''),
            'vector_top3': original[:3], 'multi_query_top3': multi_rank[:3],
            'hybrid_top3': top3, 'graph_path_count': len(paths),
            'graph_paths': paths[:4], 'anchors': anchors,
            'hits_at_3': len(hits),
            'all_expected_at_3': (len(hits)==len(expected)) if expected else None,
            'evidence_recall_at_3': len(hits)/len(expected) if expected else None,
            'latency_s': round(time.perf_counter()-t0, 3),
            'retrieved_evidence': [{'id':cid, 'heading':sources[cid]['heading'],
                                    'text':sources[cid]['evidence']}
                                   for cid in top3 if cid in sources],
        }



## 5 — Run 5 questions first, then 20 or 32

`RUN_COUNT = 5` initially. The notebook saves results after **every question** and resumes from previously saved results. Adjust `RUN_COUNT` to `20` or `32` and run this cell again; already-saved questions are skipped. Delete the checkpoint file only if you deliberately want to retest everything (for example, after changing code).

In [ ]:
RUN_COUNT = 5  # First try 5; then set to 20 or 32.
RUN_COUNT = min(RUN_COUNT, len(TESTS))
CHECKPOINT = OUTPUT_DIR / 'sigiriya_32_retrieval_checkpoint.json'

if CHECKPOINT.exists():
    previous = json.loads(CHECKPOINT.read_text(encoding='utf-8'))
    print(f'Resuming {len(previous)} saved question results.')
else:
    previous = []
by_id = {r['id']: r for r in previous}

for i, test in enumerate(TESTS[:RUN_COUNT], start=1):
    if test['id'] in by_id:
        print(f'{i:02}/{RUN_COUNT} {test["id"]}: already saved', flush=True)
        continue
    t0 = time.perf_counter()
    print(f'{i:02}/{RUN_COUNT} Running {test["id"]}...', flush=True)
    try:
        r = retrieve_one(test)
        print(f'   {r["status"]}, top3={r["hybrid_top3"]}, recall={r["evidence_recall_at_3"]}, {r["latency_s"]}s', flush=True)
    except Exception as exc:
        r = {**test, 'status':'ERROR', 'error':f'{type(exc).__name__}: {exc}',
             'hybrid_top3':[], 'graph_path_count':None, 'all_expected_at_3':None,
             'evidence_recall_at_3':None, 'latency_s':round(time.perf_counter()-t0,3)}
        print('   ERROR:', r['error'], flush=True)
    by_id[test['id']] = r
    CHECKPOINT.write_text(json.dumps(list(by_id.values()), ensure_ascii=False, indent=2, default=str), encoding='utf-8')

results = [by_id[t['id']] for t in TESTS[:RUN_COUNT] if t['id'] in by_id]
print(f'Completed or loaded {len(results)} results. Checkpoint: {CHECKPOINT}', flush=True)


## 6 — Summary

Reports Evidence Recall@3 over questions with expected source sections; this is a diagnostic retrieval measure, not a validated chatbot accuracy score. Correctness of answers requires manual review.

In [ ]:
summary = pd.DataFrame([{k:r.get(k) for k in (
    'id','type','question','answerability','status','expected_ids','vector_top3',
    'multi_query_top3','hybrid_top3','hits_at_3','evidence_recall_at_3',
    'all_expected_at_3','graph_path_count','latency_s','extractive_answer','error')}
    for r in results])
for col in ('expected_ids','vector_top3','multi_query_top3','hybrid_top3'):
    if col in summary:
        summary[col] = summary[col].apply(lambda x: ', '.join(x) if isinstance(x, list) else x)
if len(summary):
    supported = pd.DataFrame([r for r in results if r.get('expected_ids')])
    print('Tests processed:', len(results), '| errors:', sum(r.get('status')=='ERROR' for r in results))
    if not supported.empty:
        print('Mean evidence Recall@3:', round(supported['evidence_recall_at_3'].dropna().mean(),3))
        print('All expected chunks within top 3:', int(supported['all_expected_at_3'].sum()),'/',len(supported))
    display(summary[['id','type','status','expected_ids','hybrid_top3','evidence_recall_at_3','graph_path_count','extractive_answer']])

## 7 — Manual review fields and export

The output CSV includes blank columns for human assessment: **evidence_relevant**, **answer_grounded**, **complete**, and **notes**. Mark these manually after checking the complete source excerpts. A supported question with a good top-3 rank may still have an incomplete extractive answer. Unsupported/live questions should not be scored using Recall@3.

Files are saved under `results/retrieval/` and can be committed to GitHub **after you remove sensitive data if any**. No secrets are saved by this notebook.

In [ ]:
for col in ('evidence_relevant','answer_grounded','complete','review_notes'):
    summary[col] = ''
CSV_PATH = OUTPUT_DIR / 'sigiriya_32_question_retrieval_review.csv'
JSON_PATH = OUTPUT_DIR / 'sigiriya_32_question_retrieval_details.json'
summary.to_csv(CSV_PATH, index=False, encoding='utf-8-sig')
JSON_PATH.write_text(json.dumps(results, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
print('Saved CSV:', CSV_PATH)
print('Saved JSON:', JSON_PATH)

## 8 — Optional: ask your own question (read-only)

This cell retrieves results for a custom question without altering the 32-question evaluation or its metrics. The expected IDs are intentionally blank, so gold metrics are not computed. This is **not** a SinLlama response.

In [ ]:
CUSTOM_QUESTION = 'සීගිරියේ කැටපත් පවුරේ ඇති සටහන්වල වැදගත්කම කුමක්ද?'
custom = retrieve_one({'id':'CUSTOM','type':'custom','question':CUSTOM_QUESTION,
                       'expected_ids':[], 'answerability':'unknown'})
print('Status:', custom['status'])
print('Extractive answer:', custom['extractive_answer'])
print('Top3:', custom['hybrid_top3'])
print('Graph paths:', custom['graph_path_count'])
for ev in custom['retrieved_evidence']:
    print(f"\n{ev['id']} — {ev['heading']}\n{ev['text'][:700]}")

## 9 — Close Neo4j connection

Run when finished (or restart the notebook kernel).

In [ ]:
if 'driver' in globals():
    driver.close()
    del driver
print('Neo4j connection closed. Model stays loaded until the kernel is restarted.')

### How to interpret speed

The first E5 load is comparatively costly. Each *new* question still requires several vector queries, possible multi-query decomposition, graph traversal and source-backed answer selection. Repeated identical E5 inputs are cached, but the notebook **does not skip** essential graph-search stages. A slow individual question may indicate expensive database traversal; the per-question timing and checkpoint help identify which question. If the source scripts change or you revise retrieval settings, delete `results/retrieval/sigiriya_32_retrieval_checkpoint.json` before rerunning so results are not mixed.